# Lanjutan GPU: Mengejar F1 XLM-R Large di Reddit

Notebook ini melanjutkan riset **Ensemble Learning untuk Deteksi Sarkasme Lintas Platform Berbahasa Indonesia** (Buku Monograf 3), setelah seluruh eksperimen di CPU (lihat catatan-catatan sebelumnya) berhasil mengalahkan F1 resmi XLM-R Large di **Twitter** (0,7721 vs 0,7692) tapi mentok di **Reddit** (rekor 0,5848 vs target 0,6274) meski sudah dicoba 5 pendekatan berbeda.

**Hipotesis:** kelima percobaan CPU memakai model kecil (DistilBERT-Indonesian, 66M parameter). Tabel resmi paper (Suhartono et al., 2024) menunjukkan model yang menang (XLM-R Large, 550M parameter) jauh lebih besar. Di sini kita coba model yang jauh lebih besar (IndoBERT Base/Large, atau XLM-R sendiri) dengan GPU gratis Colab, **dikombinasikan dengan dua teknik yang sudah terbukti berhasil**: threshold tuning dan blend dengan ensemble klasik (teknik blend inilah yang memenangkan Twitter).

**Cara pakai:** Runtime → Change runtime type → pilih **GPU** (T4 sudah cukup). Lalu Runtime → Run all. Ganti nilai di sel **KONFIGURASI** di bawah untuk mencoba model lain.

## 1. Setup & Instalasi

In [1]:
import torch
print('GPU tersedia:', torch.cuda.is_available())
print('Nama GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(tidak ada -- ganti runtime ke GPU!)')

GPU tersedia: True
Nama GPU: Tesla T4


In [2]:
!pip install -q transformers accelerate sentencepiece fasttext scikit-learn pandas numpy datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 691.2 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Konfigurasi -- ubah di sini untuk bereksperimen

| Variabel | Pilihan yang masuk akal |
|---|---|
| `MODEL_NAME` | `indobenchmark/indobert-base-p1`, `indobenchmark/indobert-large-p1`, `xlm-roberta-base`, `xlm-roberta-large`, `cahya/distilbert-base-indonesian` |
| `PLATFORM` | `"reddit"` (target utama, F1 resmi 0,6274) atau `"twitter"` (target 0,7692, sudah menang di CPU) |
| `USE_JOINT_TRAINING` | `True` = latih pakai gabungan Twitter+Reddit (sudah dicoba di CPU, hasil negatif -- tapi mungkin beda dengan model besar) |
| `EPOCHS` | GPU jauh lebih cepat, aman coba 6-10 epoch (di CPU cuma sanggup 3-7) |

In [3]:
# ======================= KONFIGURASI =======================
MODEL_NAME = "xlm-roberta-base"   # coba juga xlm-roberta-large kalau waktu/VRAM cukup
PLATFORM = "reddit"                                # "reddit" atau "twitter"
USE_JOINT_TRAINING = False                          # True = gabung train Twitter+Reddit
MAX_LEN = 96
BATCH_SIZE = 8                                     # tetap 8 (sudah pas, jangan naikkan, VRAM large lebih boros)
LR = 1e-5           # <-- ganti dari 2e-5 (large butuh LR lebih kecil)
EPOCHS = 12          # <-- ganti dari 6 (LR lebih kecil = konvergen lebih lambat, kasih epoch lebih banyak)
SEED = 41
# =============================================================

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| Model:", MODEL_NAME, "| Platform:", PLATFORM)

Device: cuda | Model: xlm-roberta-base | Platform: reddit


## 3. Unduh Dataset Resmi (sama persis dgn yang dipakai paper IdSarcasm)

In [4]:
from datasets import load_dataset
import pandas as pd

tw_raw = load_dataset("w11wo/twitter_indonesia_sarcastic")
rd_raw = load_dataset("w11wo/reddit_indonesia_sarcastic")

tw = {s: tw_raw[s].to_pandas().rename(columns={"tweet": "text"})[["text", "label"]] for s in ["train", "validation", "test"]}
rd = {s: rd_raw[s].to_pandas()[["text", "label"]] for s in ["train", "validation", "test"]}

for name, d in [("twitter", tw), ("reddit", rd)]:
    print(name, {s: len(d[s]) for s in d})

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

data/train.csv: reconstructing file:   0%|          |  0.00B /  226kB            

data/train.csv: downloading bytes:           |  0.00B            

data/validation.csv: reconstructing file:   0%|          |  0.00B / 32.6kB            

data/validation.csv: downloading bytes:           |  0.00B            

...

data/test.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9881 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1411 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2824 [00:00<?, ? examples/s]

twitter {'train': 1878, 'validation': 268, 'test': 538}
reddit {'train': 9881, 'validation': 1411, 'test': 2824}


In [5]:
DATA = {"twitter": tw, "reddit": rd}[PLATFORM]
OTHER = {"twitter": rd, "reddit": tw}[PLATFORM]   # platform lain, dipakai kalau USE_JOINT_TRAINING

if USE_JOINT_TRAINING:
    train_df = pd.concat([DATA["train"], OTHER["train"]], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(f"Train gabungan: {len(train_df)} baris ({PLATFORM}={len(DATA['train'])} + lainnya={len(OTHER['train'])})")
else:
    train_df = DATA["train"]
valid_df, test_df = DATA["validation"], DATA["test"]
print(f"train={len(train_df)}  val={len(valid_df)}  test={len(test_df)}  | sarkastis di train: {int((train_df.label==1).sum())}")

train=9881  val=1411  test=2824  | sarkastis di train: 2470


## 4. Fine-Tuning Model Besar (GPU)

In [6]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import numpy as np, time, json

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.enc = tokenizer(list(texts), padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

train_ds = SarcasmDataset(train_df["text"], train_df["label"], tokenizer)
valid_ds = SarcasmDataset(valid_df["text"], valid_df["label"], tokenizer)
test_ds = SarcasmDataset(test_df["text"], test_df["label"], tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06 * total_steps), num_training_steps=total_steps)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def evaluate(loader):
    model.eval()
    all_preds, all_labels, all_proba = [], [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            out = model(**{k: v.to(device) for k, v in batch.items()})
            proba = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
            preds = out.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist()); all_labels.extend(labels.numpy().tolist()); all_proba.extend(proba.tolist())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds).tolist()
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm}, np.array(all_proba), np.array(all_labels)

best_f1, best_state = -1, None
t_start = time.time()
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        labels = batch.pop("labels").to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**inputs, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        if (step + 1) % 50 == 0:
            print(f"  epoch {epoch+1}/{EPOCHS} step {step+1}/{len(train_loader)} loss={total_loss/(step+1):.4f}")
    val_m, val_proba, val_labels = evaluate(valid_loader)
    print(f"[epoch {epoch+1}] {time.time()-t0:.0f}s -> val_f1={val_m['f1']:.4f} val_acc={val_m['accuracy']:.4f}")
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_val_proba, best_val_labels = val_proba, val_labels
        print(f"    -> checkpoint terbaik baru (val_f1={best_f1:.4f})")
print(f"Total waktu training: {(time.time()-t_start)/60:.1f} menit")
model.load_state_dict(best_state)

  epoch 1/12 step 50/1236 loss=0.7455
  epoch 1/12 step 100/1236 loss=0.7256
  epoch 1/12 step 150/1236 loss=0.6921
  epoch 1/12 step 200/1236 loss=0.6719
  epoch 1/12 step 250/1236 loss=0.6534
  epoch 1/12 step 300/1236 loss=0.6383
  epoch 1/12 step 350/1236 loss=0.6222
  epoch 1/12 step 400/1236 loss=0.6185
  epoch 1/12 step 450/1236 loss=0.6126
  epoch 1/12 step 500/1236 loss=0.6088
  epoch 1/12 step 550/1236 loss=0.6028
  epoch 1/12 step 600/1236 loss=0.5996
  epoch 1/12 step 650/1236 loss=0.5955
  epoch 1/12 step 700/1236 loss=0.5929
  epoch 1/12 step 750/1236 loss=0.5858
  epoch 1/12 step 800/1236 loss=0.5819
  epoch 1/12 step 850/1236 loss=0.5775
  epoch 1/12 step 900/1236 loss=0.5735
  epoch 1/12 step 950/1236 loss=0.5679
  epoch 1/12 step 1000/1236 loss=0.5651
  epoch 1/12 step 1050/1236 loss=0.5615
  epoch 1/12 step 1100/1236 loss=0.5564
  epoch 1/12 step 1150/1236 loss=0.5566
  epoch 1/12 step 1200/1236 loss=0.5537
[epoch 1] 286s -> val_f1=0.2617 val_acc=0.7760
    -> checkp

<All keys matched successfully>

## 5. Evaluasi Test + Threshold Tuning

In [8]:
def metrics_at(y_true, proba1, thr):
    pred = (proba1 >= thr).astype(int)
    acc = accuracy_score(y_true, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "cm": confusion_matrix(y_true, pred).tolist()}

test_m, test_proba, test_labels = evaluate(test_loader)
print("F1 test (threshold=0.5):", round(test_m["f1"], 4))

best_thr = max(np.arange(0.20, 0.81, 0.02), key=lambda t: metrics_at(best_val_labels, best_val_proba, t)["f1"])
tuned_test_m = metrics_at(test_labels, test_proba, best_thr)
print(f"F1 test setelah threshold tuning (thr={best_thr:.2f}):", round(tuned_test_m["f1"], 4))

F1 test (threshold=0.5): 0.5877
F1 test setelah threshold tuning (thr=0.20): 0.5935


## 6. Blend dengan Ensemble Klasik

Teknik yang berhasil mengalahkan XLM-R Large di Twitter (CPU). Base learner klasik di-*refit* hanya di `train` (bukan train+val) supaya prediksinya di validation set benar-benar out-of-sample -- penting untuk menghindari kebocoran data saat menuning bobot blend.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
import fasttext, tempfile, uuid, os as _os

class FastTextClassifier:
    def __init__(self, epoch=25, lr=0.5, wordNgrams=2, dim=100, minCount=1):
        self.epoch, self.lr, self.wordNgrams, self.dim, self.minCount = epoch, lr, wordNgrams, dim, minCount
    def fit(self, X, y):
        self.classes_ = np.array(sorted(set(y)))
        p = _os.path.join(tempfile.gettempdir(), f"ft_{uuid.uuid4().hex}.txt")
        with open(p, "w", encoding="utf-8") as f:
            for t, l in zip(X, y):
                f.write(f"__label__{l} " + str(t).replace(chr(10), " ") + "\n")
        self.model_ = fasttext.train_supervised(input=p, epoch=self.epoch, lr=self.lr, wordNgrams=self.wordNgrams, dim=self.dim, minCount=self.minCount, verbose=0)
        _os.remove(p)
        return self
    def predict_proba(self, X):
        labels, probs = self.model_.predict([str(t).replace(chr(10), " ") for t in X], k=len(self.classes_))
        out = np.zeros((len(X), len(self.classes_)))
        for i, (lb, pr) in enumerate(zip(labels, probs)):
            for l, p in zip(lb, pr):
                out[i, np.where(self.classes_ == int(l.replace("__label__", "")))[0][0]] = p
        s = out.sum(axis=1, keepdims=True); s[s == 0] = 1
        return out / s

In [10]:
def to01(proba, classes):
    classes = list(classes)
    if classes != [0, 1]:
        i0, i1 = classes.index(0), classes.index(1)
        proba = proba[:, [i0, i1]]
    return proba

vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=2)
vec.fit(train_df["text"].astype(str))
y_train = train_df["label"].values

def make_learners():
    return {"lr": LogisticRegression(C=10, max_iter=3000),
            "svm": SVC(C=1, kernel="linear", probability=True, random_state=SEED),
            "nb": MultinomialNB(alpha=0.1),
            "fasttext": FastTextClassifier()}
learner_order = ["lr", "svm", "nb", "fasttext"]

def get_X(kind, texts):
    return vec.transform(texts) if kind != "fasttext" else list(texts)

learners = make_learners()
val_p, test_p = {}, {}
for lbl in learner_order:
    kind = "fasttext" if lbl == "fasttext" else "tfidf"
    est = learners[lbl]
    est.fit(get_X(kind, train_df["text"].astype(str)), y_train)
    val_p[lbl] = to01(est.predict_proba(get_X(kind, valid_df["text"].astype(str))), est.classes_)
    test_p[lbl] = to01(est.predict_proba(get_X(kind, test_df["text"].astype(str))), est.classes_)
    print(f"  [{lbl}] selesai")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
texts_arr = train_df["text"].astype(str).values
meta_X = np.zeros((len(train_df), len(learner_order)))
for tr_idx, va_idx in skf.split(texts_arr, y_train):
    for j, lbl in enumerate(learner_order):
        kind = "fasttext" if lbl == "fasttext" else "tfidf"
        fresh = make_learners()[lbl]
        fresh.fit(get_X(kind, texts_arr[tr_idx]), y_train[tr_idx])
        meta_X[va_idx, j] = to01(fresh.predict_proba(get_X(kind, texts_arr[va_idx])), fresh.classes_)[:, 1]

meta_lr = LogisticRegression(max_iter=2000).fit(meta_X, y_train)
val_classical = to01(meta_lr.predict_proba(np.column_stack([val_p[l][:, 1] for l in learner_order])), meta_lr.classes_)[:, 1]
test_classical = to01(meta_lr.predict_proba(np.column_stack([test_p[l][:, 1] for l in learner_order])), meta_lr.classes_)[:, 1]
print("Stacking klasik -- val_f1:", round(metrics_at(valid_df['label'].values, val_classical, 0.5)['f1'], 4),
      "test_f1:", round(metrics_at(test_df['label'].values, test_classical, 0.5)['f1'], 4))

  [lr] selesai
  [svm] selesai
  [nb] selesai
  [fasttext] selesai
Stacking klasik -- val_f1: 0.372 test_f1: 0.4207


In [11]:
# blend model besar (fine-tuned) + ensemble klasik, bobot & threshold dituning di VALIDATION saja
y_val = valid_df["label"].values
y_test = test_df["label"].values

best = {"f1": -1}
for w in np.arange(0.0, 1.01, 0.05):
    blended_val = w * best_val_proba + (1 - w) * val_classical
    for thr in np.arange(0.20, 0.81, 0.02):
        f1 = metrics_at(y_val, blended_val, thr)["f1"]
        if f1 > best["f1"]:
            best = {"f1": f1, "w": w, "thr": thr}

blended_test = best["w"] * test_proba + (1 - best["w"]) * test_classical
blend_test_m = metrics_at(y_test, blended_test, best["thr"])
print(f"Blend terbaik: w={best['w']:.2f} thr={best['thr']:.2f} (val_f1={best['f1']:.4f})")
print("F1 TEST (blend, final):", round(blend_test_m["f1"], 4))

Blend terbaik: w=0.55 thr=0.22 (val_f1=0.6196)
F1 TEST (blend, final): 0.621


## 7. Ringkasan & Perbandingan dengan Angka Resmi

In [12]:
OFFICIAL = {"twitter": 0.7692, "reddit": 0.6274}   # XLM-R Large, Suhartono et al. (2024)
official_target = OFFICIAL[PLATFORM]

results = {
    "model": MODEL_NAME,
    "platform": PLATFORM,
    "joint_training": USE_JOINT_TRAINING,
    "epochs": EPOCHS,
    "fine_tuned_raw": test_m,
    "fine_tuned_threshold_tuned": {"threshold": float(best_thr), **tuned_test_m},
    "stacking_classical": metrics_at(y_test, test_classical, 0.5),
    "blend_final": {"w": float(best["w"]), "threshold": float(best["thr"]), **blend_test_m},
}

final_f1 = max(test_m["f1"], tuned_test_m["f1"], blend_test_m["f1"])
print(f"\n{'='*60}")
print(f"Platform     : {PLATFORM}")
print(f"Model        : {MODEL_NAME}")
print(f"F1 terbaik   : {final_f1:.4f}")
print(f"XLM-R Large  : {official_target:.4f}  (resmi, Suhartono et al. 2024)")
print(f"Status       : {'>>> MENGALAHKAN <<<' if final_f1 > official_target else 'belum mengalahkan'}")
print(f"{'='*60}")

import json
with open(f"hasil_{PLATFORM}_{MODEL_NAME.replace('/', '_')}.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nDisimpan ->", f"hasil_{PLATFORM}_{MODEL_NAME.replace('/', '_')}.json")
print("Unduh file ini dari panel Files di sebelah kiri Colab, lalu kirim balik ke Claude Code utk didokumentasikan.")


Platform     : reddit
Model        : xlm-roberta-base
F1 terbaik   : 0.6210
XLM-R Large  : 0.6274  (resmi, Suhartono et al. 2024)
Status       : belum mengalahkan

Disimpan -> hasil_reddit_xlm-roberta-base.json
Unduh file ini dari panel Files di sebelah kiri Colab, lalu kirim balik ke Claude Code utk didokumentasikan.


## 8. Langkah Selanjutnya

- **Ganti `MODEL_NAME`** di sel Konfigurasi (mis. ke `xlm-roberta-large`) dan Runtime → Run all lagi untuk membandingkan model lain -- tidak perlu ubah kode lain.
- **Ganti `PLATFORM` ke `"twitter"`** untuk mencoba memperlebar kemenangan yang sudah didapat di CPU (0,7721 vs 0,7692).
- Kalau `xlm-roberta-large` kehabisan memori GPU (OOM), turunkan `BATCH_SIZE` ke 8 atau 4.
- Simpan file `hasil_*.json` yang dihasilkan (panel Files di kiri, klik kanan → Download) dan kirim balik ke sesi Claude Code semula untuk digabungkan ke dokumentasi Buku Monograf 3.